# Association Analysis: Market-Basket Patterns and Association Rules

Association analysis discovers items or events that frequently occur together. Common applications include product bundling, cross-selling, website navigation, medical co-occurrence analysis, and content recommendation.

This notebook is self-contained and implements the Apriori algorithm using only the Python standard library.

## Learning objectives

By the end of this notebook, you should be able to:

- represent transactional data;
- calculate support, confidence, lift, leverage, and conviction;
- find frequent itemsets with Apriori;
- generate and filter association rules;
- use rules to produce simple basket recommendations; and
- recognize common interpretation pitfalls.

## 1. Transactional data

Each transaction is a set of items. Repeated copies of an item within one transaction do not affect ordinary market-basket analysis.

In [ ]:
transactions = [
    {'bread', 'milk'},
    {'bread', 'diapers', 'beer', 'eggs'},
    {'milk', 'diapers', 'beer', 'cola'},
    {'bread', 'milk', 'diapers', 'beer'},
    {'bread', 'milk', 'diapers', 'cola'},
    {'bread', 'butter'},
    {'milk', 'cereal'},
    {'bread', 'milk', 'butter'},
    {'diapers', 'beer'},
    {'bread', 'milk', 'diapers'},
]

transactions = [frozenset(t) for t in transactions]
print('Transactions:', len(transactions))
print('Unique items:', sorted(set().union(*transactions)))

## 2. Core metrics

For an itemset $X$ and rule $X \rightarrow Y$:

- **Support:** $P(X)$, the proportion of transactions containing $X$.
- **Confidence:** $P(Y \mid X) = P(X \cup Y)/P(X)$.
- **Lift:** $P(X \cup Y)/(P(X)P(Y))$. Lift above 1 indicates positive association; lift equal to 1 indicates independence.
- **Leverage:** $P(X \cup Y)-P(X)P(Y)$. This measures the absolute departure from independence.
- **Conviction:** $(1-P(Y))/(1-P(Y \mid X))$. Larger values indicate that the rule makes fewer incorrect predictions than expected under independence.

Association does not establish causation, and confidence is directional: $X \rightarrow Y$ generally differs from $Y \rightarrow X$.

In [ ]:
def support(itemset, transactions):
    itemset = frozenset(itemset)
    if not transactions:
        return 0.0
    return sum(itemset <= transaction for transaction in transactions) / len(transactions)


bread = support({'bread'}, transactions)
milk = support({'milk'}, transactions)
bread_and_milk = support({'bread', 'milk'}, transactions)
confidence_bread_to_milk = bread_and_milk / bread
lift_bread_to_milk = confidence_bread_to_milk / milk

print(f'Support(bread) = {bread:.2f}')
print(f'Support(milk) = {milk:.2f}')
print(f'Support(bread, milk) = {bread_and_milk:.2f}')
print(f'Confidence(bread -> milk) = {confidence_bread_to_milk:.3f}')
print(f'Lift(bread -> milk) = {lift_bread_to_milk:.3f}')

## 3. Apriori frequent-itemset mining

Apriori uses the downward-closure property: if an itemset is frequent, all of its subsets must also be frequent. Therefore, a candidate can be discarded as soon as one of its smaller subsets is known to be infrequent.

In [ ]:
from itertools import combinations


def apriori(transactions, min_support=0.3):
    if not 0 < min_support <= 1:
        raise ValueError('min_support must be in (0, 1]')

    transactions = [frozenset(t) for t in transactions]
    if not transactions:
        return {}

    universe = sorted(set().union(*transactions))
    support_map = {}
    previous_level = set()

    for item in universe:
        candidate = frozenset([item])
        candidate_support = support(candidate, transactions)
        if candidate_support >= min_support:
            previous_level.add(candidate)
            support_map[candidate] = candidate_support

    k = 2
    while previous_level:
        candidates = set()
        previous_list = list(previous_level)
        for left, right in combinations(previous_list, 2):
            candidate = left | right
            if len(candidate) != k:
                continue
            subsets = (frozenset(s) for s in combinations(candidate, k - 1))
            if all(subset in previous_level for subset in subsets):
                candidates.add(candidate)

        current_level = set()
        for candidate in candidates:
            candidate_support = support(candidate, transactions)
            if candidate_support >= min_support:
                current_level.add(candidate)
                support_map[candidate] = candidate_support

        previous_level = current_level
        k += 1

    return support_map


frequent_itemsets = apriori(transactions, min_support=0.3)
for itemset, itemset_support in sorted(
        frequent_itemsets.items(), key=lambda pair: (len(pair[0]), sorted(pair[0]))):
    print(f'{sorted(itemset)!s:38} support={itemset_support:.2f}')

## 4. Generate association rules

Every frequent itemset of size two or greater can be divided into a nonempty antecedent and consequent. We retain rules meeting minimum confidence and lift thresholds.

In [ ]:
def association_rules(support_map, min_confidence=0.6, min_lift=1.0):
    if not 0 <= min_confidence <= 1:
        raise ValueError('min_confidence must be in [0, 1]')

    rules = []
    for itemset, joint_support in support_map.items():
        if len(itemset) < 2:
            continue

        items = sorted(itemset)
        for antecedent_size in range(1, len(items)):
            for antecedent_tuple in combinations(items, antecedent_size):
                antecedent = frozenset(antecedent_tuple)
                consequent = itemset - antecedent
                antecedent_support = support_map[antecedent]
                consequent_support = support_map[consequent]
                confidence = joint_support / antecedent_support
                lift = confidence / consequent_support
                leverage = joint_support - antecedent_support * consequent_support
                conviction = (
                    float('inf') if confidence == 1
                    else (1 - consequent_support) / (1 - confidence)
                )

                if confidence >= min_confidence and lift >= min_lift:
                    rules.append({
                        'antecedent': antecedent,
                        'consequent': consequent,
                        'support': joint_support,
                        'confidence': confidence,
                        'lift': lift,
                        'leverage': leverage,
                        'conviction': conviction,
                    })

    return sorted(rules, key=lambda rule: (-rule['lift'], -rule['confidence'], -rule['support']))


rules = association_rules(frequent_itemsets, min_confidence=0.6, min_lift=1.0)
for rule in rules:
    left = ', '.join(sorted(rule['antecedent']))
    right = ', '.join(sorted(rule['consequent']))
    print(
        f'{left:20} -> {right:20} '
        f"support={rule['support']:.2f}  confidence={rule['confidence']:.2f}  "
        f"lift={rule['lift']:.2f}  leverage={rule['leverage']:.3f}"
    )

## 5. Interpreting an example

A rule such as `diapers -> beer` means that baskets containing diapers often also contain beer. Its metrics answer different questions:

- support asks how common the combined basket is;
- confidence asks how often beer occurs when diapers occur;
- lift compares that confidence with beer's overall popularity; and
- leverage measures the absolute increase over the count expected under independence.

A high-confidence rule can still be uninteresting when its consequent is already extremely common. Lift helps expose this base-rate problem.

In [ ]:
def describe_rule(antecedent, consequent, transactions):
    antecedent = frozenset(antecedent)
    consequent = frozenset(consequent)
    joint = antecedent | consequent
    s_a = support(antecedent, transactions)
    s_c = support(consequent, transactions)
    s_joint = support(joint, transactions)
    confidence = s_joint / s_a
    return {
        'support': s_joint,
        'confidence': confidence,
        'lift': confidence / s_c,
        'leverage': s_joint - s_a * s_c,
    }


diapers_to_beer = describe_rule({'diapers'}, {'beer'}, transactions)
beer_to_diapers = describe_rule({'beer'}, {'diapers'}, transactions)
print('diapers -> beer:', diapers_to_beer)
print('beer -> diapers:', beer_to_diapers)
print('Notice that support and lift are symmetric, but confidence is directional.')

## 6. Example: basket recommendations

Rules can form a small recommendation engine. A rule applies when its antecedent is contained in the current basket and its consequent contains something not already present.

In [ ]:
def recommend_from_rules(basket, rules, limit=5):
    basket = frozenset(basket)
    candidates = {}

    for rule in rules:
        if rule['antecedent'] <= basket:
            for item in rule['consequent'] - basket:
                score = rule['confidence'] * rule['lift']
                current = candidates.get(item)
                if current is None or score > current['score']:
                    candidates[item] = {
                        'item': item,
                        'score': score,
                        'confidence': rule['confidence'],
                        'lift': rule['lift'],
                        'because': sorted(rule['antecedent']),
                    }

    return sorted(candidates.values(), key=lambda row: -row['score'])[:limit]


basket = {'diapers'}
print('Current basket:', basket)
for recommendation in recommend_from_rules(basket, rules):
    print(recommendation)

## 7. Example beyond retail: course-topic co-occurrence

A transaction need not be a shopping basket. Here, each transaction represents the topics selected by one learner. The same analysis identifies topics that are commonly studied together.

In [ ]:
learning_paths = [
    {'python', 'statistics', 'machine-learning'},
    {'python', 'sql', 'data-visualization'},
    {'python', 'statistics', 'machine-learning', 'deep-learning'},
    {'sql', 'data-visualization', 'business-intelligence'},
    {'python', 'statistics', 'data-visualization'},
    {'machine-learning', 'deep-learning', 'nlp'},
    {'python', 'machine-learning', 'nlp'},
    {'statistics', 'machine-learning', 'experimentation'},
]

topic_itemsets = apriori(learning_paths, min_support=0.25)
topic_rules = association_rules(topic_itemsets, min_confidence=0.6, min_lift=1.05)
for rule in topic_rules[:10]:
    print(
        f"{sorted(rule['antecedent'])} -> {sorted(rule['consequent'])}: "
        f"confidence={rule['confidence']:.2f}, lift={rule['lift']:.2f}"
    )

## 8. Validation and edge cases

Small assertions are useful when implementing mining algorithms. They verify metric bounds, the Apriori downward-closure property, and expected rule behavior.

In [ ]:
assert support(set(), transactions) == 1.0
assert support({'item-that-does-not-exist'}, transactions) == 0.0
assert all(0 <= value <= 1 for value in frequent_itemsets.values())

for itemset in frequent_itemsets:
    if len(itemset) > 1:
        for subset_size in range(1, len(itemset)):
            for subset in combinations(itemset, subset_size):
                assert frozenset(subset) in frequent_itemsets

assert all(rule['antecedent'].isdisjoint(rule['consequent']) for rule in rules)
assert all(rule['lift'] >= 1.0 for rule in rules)
print('All validation checks passed.')

## 9. Practical considerations

- **Threshold selection:** A high minimum support can hide valuable niche patterns; a low threshold can produce an overwhelming number of rules.
- **Rare-item problem:** Lift can become very large for items occurring only a few times. Always inspect support alongside lift.
- **Multiple comparisons:** Mining thousands of possible rules can surface coincidences. Validate important rules on held-out or later data.
- **Time and context:** Basic association rules ignore order, quantity, price, and time. Sequential-pattern mining is more suitable when event order matters.
- **Data leakage:** Build rules only from information available before the intended recommendation or decision.
- **Operational value:** A statistically strong rule may be unusable because of inventory, margin, policy, fairness, or customer-experience constraints.

For large datasets, production libraries such as `mlxtend.frequent_patterns` provide optimized Apriori and FP-Growth implementations.

## 10. Practice exercises

1. Change `min_support` from `0.30` to `0.20`. How does the number of frequent itemsets change?
2. Find all rules with confidence at least `0.75`, then sort them by leverage instead of lift.
3. Compare `diapers -> beer` with `beer -> diapers`. Explain why their confidence differs.
4. Add three transactions containing a new rare item. Can you create a high-lift but low-support rule?
5. Modify `recommend_from_rules` so that evidence from multiple applicable rules is summed rather than taking only the strongest rule.
6. Create a new non-retail transaction dataset, such as medical symptoms, web pages visited, or movie genres, and interpret its strongest rules.

## Summary

Association analysis turns co-occurrence data into frequent itemsets and directional rules. Support measures prevalence, confidence measures conditional frequency, and lift compares a rule with the independence baseline. These metrics should be interpreted together and validated before rules are used for real decisions.